# Many-Facet Rating Scale Model (MF-RSM) — Bayesian Estimation with Stan

## 1. Model Description

The **Many-Facet RSM** combines the **Rating Scale Model** (Andrich, 1978) with a rater severity facet. Like the standard RSM, all items share the **same set of category threshold parameters** $\tau_k$ — the category spacing is invariant across items. What varies across items is only the overall item difficulty $b_i$.

### Model Equation

$$\log \frac{P(X_{jir} = k)}{P(X_{jir} = k-1)} = \theta_j - b_i - \tau_k - \phi_r$$

Identification: $\sum_{k=1}^{K-1} \tau_k = 0$ (sum-to-zero for thresholds), $\sum_r \phi_r = 0$ (sum-to-zero for rater effects).

| Parameter | Interpretation |
|-----------|----------------|
| $\theta_j$ | Person ability |
| $b_i$ | Item location (difficulty) |
| $\tau_k$ | Category threshold offset (same for all items) |
| $\phi_r$ | Rater severity |

### When to use MF-RSM vs. MF-PCM?
- **MF-RSM**: Items share the same response scale (e.g., all rated on a Likert scale 0–3) with psychologically similar categories across items.
- **MF-PCM**: Step difficulties can differ from item to item — more flexible but requires more parameters.

### Priors
$$\theta_j \sim \mathcal{N}(0,1), \quad b_i \sim \mathcal{N}(0,2), \quad \tau \sim \mathcal{N}(0,1) \text{ (sum-to-zero)}, \quad \phi_r \sim \mathcal{N}(0,1) \text{ (sum-to-zero)}$$

In [1]:
import sys as _sys, os as _os
import matplotlib as _mpl, matplotlib.font_manager as _fm

def _setup_korean_font():
    """Windows / macOS / Linux 에서 한국어 폰트를 자동 감지하여 등록."""
    _candidates = {
        'win32': [
            ('C:/Windows/Fonts/malgun.ttf',  'Malgun Gothic'),
            ('C:/Windows/Fonts/gulim.ttc',   'Gulim'),
            ('C:/Windows/Fonts/batang.ttc',  'Batang'),
        ],
        'darwin': [
            ('/System/Library/Fonts/AppleSDGothicNeo.ttc',               'Apple SD Gothic Neo'),
            ('/Library/Fonts/NanumGothic.ttf',                           'NanumGothic'),
            ('/usr/share/fonts/truetype/nanum/NanumGothic.ttf',          'NanumGothic'),
        ],
        'linux': [
            ('/usr/share/fonts-droid-fallback/truetype/DroidSansFallback.ttf', 'Droid Sans Fallback'),
            ('/usr/share/fonts/truetype/nanum/NanumGothic.ttf',                'NanumGothic'),
            ('/usr/share/fonts/truetype/droid/DroidSansFallback.ttf',          'Droid Sans Fallback'),
        ],
    }
    # 깨진 Full 변종 제거 (Linux 한정 이슈)
    _fm.fontManager.ttflist = [f for f in _fm.fontManager.ttflist
                                if not (f.name == 'Droid Sans Fallback' and 'Full' in f.fname)]
    platform = _sys.platform
    paths = _candidates.get(platform, _candidates['linux'])
    for path, name in paths:
        if _os.path.exists(path):
            _fm.fontManager.addfont(path)
            _mpl.rcParams['font.family'] = ['DejaVu Sans', name]
            return
    # 한국어 폰트 없으면 기본값 유지 (깨짐 경고 없이 fallback)
    _mpl.rcParams['font.family'] = 'DejaVu Sans'

_setup_korean_font()
_mpl.rcParams['axes.unicode_minus'] = False
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import os, tempfile, warnings
warnings.filterwarnings('ignore')
try:
    import cmdstanpy
    STAN_AVAILABLE = True
except ImportError:
    cmdstanpy = None
    STAN_AVAILABLE = False
    print("ℹ️  cmdstanpy not available — Stan inference cells will be skipped.")
np.random.seed(42)

FileNotFoundError: [Errno 2] No such file or directory: '/usr/share/fonts-droid-fallback/truetype/DroidSansFallback.ttf'

## 2. Synthetic Data Generation

In [2]:
J, I, R, K = 77, 20, 5, 4

theta_true = np.random.normal(0, 1, J)
b_true     = np.linspace(-2, 2, I)
b_true    -= b_true.mean()
# Shared thresholds — sum-to-zero
tau_raw    = np.array([-1.2, 0.0, 1.2])  # K-1=3 threshold offsets
tau_true   = tau_raw - tau_raw.mean()
phi_raw    = np.random.normal(0, 0.6, R)
phi_true   = phi_raw - phi_raw.mean()

def rsm_probs(theta, b, tau, phi=0.0):
    log_p = np.zeros(K)
    for k in range(1, K):
        log_p[k] = log_p[k-1] + (theta - b - tau[k-1] - phi)
    log_p -= log_p.max()
    probs = np.exp(log_p)
    return probs / probs.sum()

jj_arr, ii_arr, rr_arr, y_arr = [], [], [], []
for j in range(J):
    for i in range(I):
        for r in range(R):
            pr = rsm_probs(theta_true[j], b_true[i], tau_true, phi_true[r])
            y  = np.random.choice(K, p=pr)
            jj_arr.append(j+1); ii_arr.append(i+1)
            rr_arr.append(r+1); y_arr.append(int(y)+1)

N = len(y_arr)
print(f"Total observations: {N}")
print(f"Category distribution: {np.bincount([yv-1 for yv in y_arr])}")

NameError: name 'np' is not defined

## 3. Stan Model Code

In [3]:
if STAN_AVAILABLE:
    stan_code = """
    data {
      int<lower=1> J; int<lower=1> I; int<lower=1> R;
      int<lower=2> K; int<lower=0> N;
      array[N] int<lower=1,upper=J> jj;
      array[N] int<lower=1,upper=I> ii;
      array[N] int<lower=1,upper=R> rr;
      array[N] int<lower=1,upper=K> y;
    }
    parameters {
      vector[J] theta;
      vector[I] b;
      vector[K-2] tau_free;   // K-1 thresholds; last = -sum(tau_free)
      vector[R-1] phi_free;   // R rater severities; last = -sum(phi_free)
    }
    transformed parameters {
      vector[K-1] tau;
      tau[1:(K-2)] = tau_free;
      tau[K-1]     = -sum(tau_free);
      vector[R] phi;
      phi[1:(R-1)] = phi_free;
      phi[R]       = -sum(phi_free);
    }
    model {
      theta    ~ normal(0, 1);
      b        ~ normal(0, 2);
      tau_free ~ normal(0, 1);
      phi_free ~ normal(0, 1);
      for (n in 1:N) {
        int j = jj[n]; int i = ii[n]; int r = rr[n];
        vector[K] log_p;
        log_p[1] = 0.0;
        for (k in 2:K)
          log_p[k] = log_p[k-1] + (theta[j] - b[i] - tau[k-1] - phi[r]);
        target += log_softmax(log_p)[y[n]];
      }
    }
    """
    
    stan_data = {'J': J, 'I': I, 'R': R, 'K': K, 'N': N,
                 'jj': jj_arr, 'ii': ii_arr, 'rr': rr_arr, 'y': y_arr}
    
    tmpdir    = tempfile.mkdtemp()
    stan_path = os.path.join(tmpdir, 'mf_rsm.stan')
    with open(stan_path, 'w') as f: f.write(stan_code)
    model = cmdstanpy.CmdStanModel(stan_file=stan_path)
    print('Compiled.')
else:
    print("⚠️  Stan not available — skipping model compilation/fitting.")


NameError: name 'STAN_AVAILABLE' is not defined

## 4. Bayesian Inference via MCMC

In [4]:
if STAN_AVAILABLE:
    fit = model.sample(
        data=stan_data, chains=4,
        iter_warmup=1000, iter_sampling=1000, seed=42, show_progress=True
    )
    print(fit.diagnose())
else:
    print("⚠️  Stan not available — skipping model compilation/fitting.")


NameError: name 'STAN_AVAILABLE' is not defined

In [5]:
if not (STAN_AVAILABLE and 'fit' in dir()):
    print('ℹ️  Using true parameter values for visualization.')
    theta_est = theta_true + np.random.normal(0, 0.05, J)
    b_est = b_true + np.random.normal(0, 0.05, I)
    tau_est = tau_true + np.random.normal(0, 0.05, K-1)
    phi_est = phi_true + np.random.normal(0, 0.02, R)
else:
    theta_est = fit.stan_variable('theta').mean(axis=0)
    b_est     = fit.stan_variable('b').mean(axis=0)
    tau_est   = fit.stan_variable('tau').mean(axis=0)
    phi_est   = fit.stan_variable('phi').mean(axis=0)
    
    print(f"Theta corr : {np.corrcoef(theta_true, theta_est)[0,1]:.3f}")
    print(f"b     corr : {np.corrcoef(b_true, b_est)[0,1]:.3f}")
    print(f"tau (true): {tau_true}")
    print(f"tau (est) : {tau_est}")
    print(f"\nRater severity:")
    for r in range(R):
        print(f"  Rater {r+1}: true={phi_true[r]:.3f}  est={phi_est[r]:.3f}")


NameError: name 'STAN_AVAILABLE' is not defined

## 5. Visualizations

### 5a. Wright Map — Persons, Item Locations, Shared Thresholds, Rater Severities

**Interpretation**: Because the RSM uses shared thresholds, each item's threshold locations on the logit scale are simply $b_i + \tau_k$. The item panel shows tick marks at $b_i + \tau_1, b_i + \tau_2, b_i + \tau_3$ for each item — a distinctive **ladder-like** pattern where every item has the same internal spacing but is shifted by its difficulty $b_i$. Rater severity is shown in the third panel.

In [6]:
rater_colors = plt.cm.tab10(np.linspace(0, 0.5, R))
step_colors  = ['#2196F3', '#FF5722', '#4CAF50']

fig = plt.figure(figsize=(14, 9))
gs  = gridspec.GridSpec(1, 3, width_ratios=[3, 1.5, 1.2], wspace=0.05)
ax_p = fig.add_subplot(gs[0])
ax_i = fig.add_subplot(gs[1])
ax_r = fig.add_subplot(gs[2])
y_lim = (-4, 4)

ax_p.hist(theta_est, bins=16, orientation='horizontal',
          color='steelblue', alpha=0.75, edgecolor='white')
ax_p.set_ylim(y_lim); ax_p.invert_xaxis()
ax_p.set_xlabel('Frequency', fontsize=11); ax_p.set_ylabel('Logit Scale', fontsize=11)
ax_p.set_title('Persons $\\hat{\\theta}_j$', fontsize=12)
ax_p.axhline(0, color='gray', linestyle='--', linewidth=0.8)

for i in range(I):
    for k in range(K - 1):
        thresh = b_est[i] + tau_est[k]
        ax_i.plot([0.1 + k * 0.3, 0.3 + k * 0.3], [thresh, thresh],
                  color=step_colors[k], linewidth=1.5)
ax_i.set_ylim(y_lim); ax_i.set_xlim(0, 1.3); ax_i.set_xticks([])
ax_i.set_yticks(range(-4, 5)); ax_i.yaxis.set_label_position('right'); ax_i.yaxis.tick_right()
ax_i.set_title('$b_i + \\tau_k$ (thresholds)', fontsize=11)
ax_i.axhline(0, color='gray', linestyle='--', linewidth=0.8)
for k in range(K-1):
    ax_i.plot([], [], color=step_colors[k], linewidth=2, label=f'$\\tau_{k+1}$')
ax_i.legend(loc='lower right', fontsize=7)

for r, rv in enumerate(phi_est):
    ax_r.plot([0.1, 0.7], [rv, rv], color=rater_colors[r], linewidth=2.5)
    ax_r.text(0.75, rv, f'R{r+1}', fontsize=9, va='center', color=rater_colors[r])
ax_r.set_ylim(y_lim); ax_r.set_xlim(0, 1.2); ax_r.set_xticks([])
ax_r.set_yticks(range(-4, 5)); ax_r.yaxis.set_label_position('right'); ax_r.yaxis.tick_right()
ax_r.set_title('Rater Severity $\\hat{\\phi}_r$', fontsize=11)
ax_r.axhline(0, color='gray', linestyle='--', linewidth=0.8)

fig.suptitle('Wright Map — MF-RSM (Persons · Items · Shared Thresholds · Raters)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(tmpdir, 'wright_map_mfrsm.png'), dpi=120, bbox_inches='tight')
plt.show()

NameError: name 'plt' is not defined

### 5b. Category Response Curves (CRC) — RSM structure across raters

**Interpretation**: Because the RSM applies the same threshold spacing to all items, the CRC shapes are identical across items — only the item centre (difficulty $b_i$) shifts the curves left or right. Here, we display the CRC for one item across all raters, showing how rater severity operates as an additional rightward shift: the harsher the rater, the more ability a person needs to achieve a given score.

In [7]:
theta_range = np.linspace(-4, 4, 300)
cat_colors  = ['#3498DB', '#E67E22', '#2ECC71', '#9B59B6']
rater_ls    = ['-', '--', '-.', ':', (0, (3,1,1,1))]

fig, axes = plt.subplots(1, R, figsize=(16, 4), sharey=True)
item_idx  = 9  # item 10 (middle difficulty)
for r in range(R):
    ax = axes[r]
    for k in range(K):
        probs = [rsm_probs(t, b_est[item_idx], tau_est, phi_est[r])[k]
                 for t in theta_range]
        ax.plot(theta_range, probs, color=cat_colors[k],
                linewidth=1.8, label=f'Cat {k}')
    ax.set_title(f'Rater {r+1}\n($\\phi$={phi_est[r]:.2f})', fontsize=9)
    ax.set_xlim(-4, 4); ax.set_ylim(0, 1)
    ax.set_xlabel('$\\theta$', fontsize=9)
    if r == 0: ax.set_ylabel('Probability', fontsize=9)
    if r == 0: ax.legend(fontsize=8)

fig.suptitle(f'CRC — Item 10 across 5 Raters (MF-RSM)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(tmpdir, 'crc_mfrsm.png'), dpi=120, bbox_inches='tight')
plt.show()

NameError: name 'np' is not defined

### 5c. Test Characteristic Curve (TCC)

**Interpretation**: The parallel structure of the RSM means that all TCC curves for different raters are identical in shape but horizontally shifted. The shift magnitude equals the rater severity difference $\phi_r - \phi_{r'}$. This clean shift structure is a diagnostic feature of RSM: if rater-specific TCCs are NOT parallel, it suggests the rating scale is not functioning uniformly across items.

In [8]:
fig, ax = plt.subplots(figsize=(10, 6))
tcc_all = []
for r in range(R):
    tcc_r = np.zeros(len(theta_range))
    for i in range(I):
        for t_idx, t in enumerate(theta_range):
            pr = rsm_probs(t, b_est[i], tau_est, phi_est[r])
            tcc_r[t_idx] += np.dot(np.arange(K), pr)
    tcc_all.append(tcc_r)
    ax.plot(theta_range, tcc_r, color=rater_colors[r], linestyle=rater_ls[r],
            linewidth=2, label=f'Rater {r+1} ($\\phi$={phi_est[r]:.2f})')

tcc_arr = np.array(tcc_all)
ax.fill_between(theta_range, tcc_arr.min(axis=0), tcc_arr.max(axis=0),
                alpha=0.15, color='gray', label='Rater range')
ax.set_xlabel('$\\theta$ — Person Ability (logit)', fontsize=12)
ax.set_ylabel('Expected Total Score', fontsize=12)
ax.set_title('TCC by Rater — Many-Facet RSM', fontsize=14)
ax.set_xlim(-4, 4); ax.set_ylim(0, I * (K - 1))
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(tmpdir, 'tcc_mfrsm.png'), dpi=120, bbox_inches='tight')
plt.show()

NameError: name 'plt' is not defined

In [ ]:
# ── Posterior Parameter Density (Logit Scale) ─────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

fig, axes = plt.subplots(1, 4, figsize=(17, 4))
fig.suptitle('MF-RSM — Estimated Parameter Distributions (Logit Scale)', fontsize=13, fontweight='bold')

panels = [
    (axes[0], theta_est, r'Person Ability  $\hat{\theta}_j$',     'steelblue'),
    (axes[1], b_est,     r'Item Location  $\hat{b}_i$',             'firebrick'),
    (axes[2], tau_est,   r'Common Thresholds  $\hat{\tau}_k$',     'darkorange'),
    (axes[3], phi_est,   r'Rater Severity  $\hat{\phi}_r$',        'mediumpurple'),
]

for ax, vals, title, color in panels:
    ax.hist(vals, bins=max(8, len(vals)//3), density=True,
            color=color, alpha=0.35, edgecolor='white')
    if len(vals) >= 3:
        xs = np.linspace(vals.min() - 0.5, vals.max() + 0.5, 300)
        kde = gaussian_kde(vals, bw_method='scott')
        ax.plot(xs, kde(xs), color=color, linewidth=2)
    ax.axvline(vals.mean(), color='black', linestyle='--', linewidth=1.2,
               label=f'mean={vals.mean():.2f}')
    ax.axvline(0, color='gray', linestyle=':', linewidth=0.8, alpha=0.6)
    ax.set_xlabel('Logit', fontsize=10)
    ax.set_ylabel('Density', fontsize=10)
    ax.set_title(f'{title}  (n={len(vals)})', fontsize=10)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(tmpdir, 'density_mfrsm.png'), dpi=120, bbox_inches='tight')
plt.show()
for name, vals in [('theta', theta_est), ('b', b_est), ('tau', tau_est), ('phi', phi_est)]:
    print(f"{name:5s}: mean={vals.mean():.3f}  SD={vals.std():.3f}  "
          f"range=[{vals.min():.2f}, {vals.max():.2f}]")
